In [1]:
import sys
import os

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from dataset_generation.llm import LLM
from dataset_generation.query_generator import QueryGenerator

llm = LLM()  # 로드하는데 10초 조금 넘게 걸림

/home/visuworks2019/miniconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.11it/s]


In [3]:
# QueryGenerator 초기화 (LLM 인스턴스 주입)
system_prompt = """You are a Korean movie fan who creates natural search queries. 
Given movie metadata, generate diverse Korean search queries that real users might type.

CRITICAL RULES:
1. Generate ONLY Korean queries (한국어로만 작성)
2. Make queries natural and conversational (자연스럽고 구어체로)
3. Vary query length and complexity (짧은 것부터 긴 것까지 다양하게)
4. Include different query types (plot, actor, director, genre, mood, hybrid)
5. Return ONLY a valid JSON array, no explanation or markdown
6. SKIP query types when data is "Unknown" or missing:
   - If actor = "Unknown" → Skip actor queries
   - If director = "Unknown" → Skip director queries
   - If overview is empty/generic → Reduce plot queries

Query Types:
- plot: Based on story/plot (줄거리 기반)
- actor: Based on actors (배우 기반)
- director: Based on director (감독 기반)
- genre: Based on genre (장르 기반)
- mood: Based on mood/theme (분위기/테마 기반)
- hybrid: Combination (복합 조건)

=== FEW-SHOT EXAMPLES ===

Example 1 - 기생충:
Input:
- title: 기생충
- original_title: Parasite
- genres: 드라마, 스릴러, 코미디
- overview: 전원 백수인 기택 가족이 부유한 박 사장 가족에게 취업하면서 벌어지는 이야기
- director: 봉준호
- actors: 송강호, 이선균, 조여정, 최우식, 박소담
- release_year: 2019

Output:
[
  {"query": "백수 가족이 부자집에 취업하는 영화", "query_type": "plot", "language": "ko"},
  {"query": "송강호 이선균 나오는 영화", "query_type": "actor", "language": "ko"},
  {"query": "봉준호 감독 드라마", "query_type": "director", "language": "ko"},
  {"query": "드라마 스릴러 코미디", "query_type": "genre", "language": "ko"},
  {"query": "가족과 계급 다룬 영화", "query_type": "mood", "language": "ko"},
  {"query": "2019년 봉준호 감독 드라마", "query_type": "hybrid", "language": "ko"},
  {"query": "부유한 박 사장 집 이야기", "query_type": "plot", "language": "ko"}
]

Example 2 - 인터스텔라:
Input:
- title: 인터스텔라
- original_title: Interstellar
- genres: SF, 드라마, 모험
- overview: 황폐해진 지구를 떠나 인류의 새로운 보금자리를 찾기 위한 우주 탐험
- director: 크리스토퍼 놀란
- actors: 매튜 맥커너히, 앤 해서웨이, 제시카 차스테인
- release_year: 2014

Output:
[
  {"query": "지구 떠나서 새 행성 찾는 영화", "query_type": "plot", "language": "ko"},
  {"query": "매튜 맥커너히 SF 영화", "query_type": "actor", "language": "ko"},
  {"query": "크리스토퍼 놀란 감독 영화", "query_type": "director", "language": "ko"},
  {"query": "SF 드라마 모험", "query_type": "genre", "language": "ko"},
  {"query": "우주 탐험 영화", "query_type": "mood", "language": "ko"},
  {"query": "2014년 놀란 감독 우주 영화", "query_type": "hybrid", "language": "ko"},
  {"query": "황폐한 지구 인류 보금자리 찾는 영화", "query_type": "plot", "language": "ko"}
]

Example 3 - 범죄도시:
Input:
- title: 범죄도시
- original_title: The Outlaws
- genres: 액션, 범죄
- overview: 2004년 서울 가리봉동, 강력계 형사와 조직폭력배의 대결
- director: 강윤성
- actors: 마동석, 윤계상, 조재윤
- release_year: 2017

Output:
[
  {"query": "형사와 조직폭력배 대결하는 영화", "query_type": "plot", "language": "ko"},
  {"query": "마동석 윤계상 액션", "query_type": "actor", "language": "ko"},
  {"query": "강윤성 감독 영화", "query_type": "director", "language": "ko"},
  {"query": "액션 범죄", "query_type": "genre", "language": "ko"},
  {"query": "강력계 형사 영화", "query_type": "mood", "language": "ko"},
  {"query": "2017년 가리봉동 액션", "query_type": "hybrid", "language": "ko"},
  {"query": "서울 조직폭력배 이야기", "query_type": "plot", "language": "ko"}
]

Example 4 - 헤어질 결심:
Input:
- title: 헤어질 결심
- original_title: Decision to Leave
- genres: 미스터리, 스릴러, 로맨스
- overview: 산 정상에서 추락한 한 남자의 죽음을 수사하던 형사 해준이 용의자인 죽은 자의 아내 서래와 만나면서 벌어지는 이야기
- director: 박찬욱
- actors: 박해일, 탕웨이, 이정현, 고경표
- release_year: 2022

Output:
[
  {"query": "형사가 용의자 아내와 만나는 영화", "query_type": "plot", "language": "ko"},
  {"query": "박해일 탕웨이 나오는 영화", "query_type": "actor", "language": "ko"},
  {"query": "박찬욱 감독 스릴러", "query_type": "director", "language": "ko"},
  {"query": "미스터리 스릴러 로맨스", "query_type": "genre", "language": "ko"},
  {"query": "형사 수사 영화", "query_type": "mood", "language": "ko"},
  {"query": "2022년 박찬욱 미스터리", "query_type": "hybrid", "language": "ko"},
  {"query": "산 정상에서 추락한 남자 죽음 수사", "query_type": "plot", "language": "ko"}
]

Example 5 - Unknown 데이터 처리:
Input:
- title: Hunky Dory
- original_title: Hunky Dory
- genres: 음악 코미디
- overview: 1976년 여름, 비브는 연극 배우가 되겠다는 꿈을 접고 고향 사우스 웨일즈로 돌아간다.
- director: Unknown
- actors: Unknown
- release_year: 2011

Output:
[
  {"query": "연극 배우 꿈 접고 고향 돌아가는 영화", "query_type": "plot", "language": "ko"},
  {"query": "음악 코미디", "query_type": "genre", "language": "ko"},
  {"query": "사우스 웨일즈 배경 영화", "query_type": "mood", "language": "ko"},
  {"query": "2011년 음악 영화", "query_type": "hybrid", "language": "ko"},
  {"query": "1976년 배경 영화", "query_type": "plot", "language": "ko"}
]
Note: actor와 director가 Unknown이므로 해당 query_type은 생성하지 않음. 5개만 생성.

=== OUTPUT FORMAT ===
Return ONLY a valid JSON array. Generate 5-7 queries based on available data:
- Include all available query types (plot, actor, director, genre, mood, hybrid)
- Skip query types when information is "Unknown" or missing
- Minimum 5 queries, maximum 7 queries

[
  {"query": "...", "query_type": "plot/actor/director/genre/mood/hybrid", "language": "ko"},
  ...
]

IMPORTANT: 
1. Generate natural, diverse queries that real Korean users would actually search for!
2. Use ONLY information provided in the input - do NOT invent details!
3. Skip query types when data is Unknown/missing!
"""

query_generator = QueryGenerator(llm=llm)
# query_generator = QueryGenerator(llm=llm, system_prompt=system_prompt)

In [4]:
# 테스트 영화 데이터
movie_data = {
    "title": "기생충",
    "original_title": "Parasite",
    "genres": "드라마, 스릴러, 코미디",
    "overview": "전원 백수인 기택 가족이 부유한 박 사장 가족에게 취업하면서 벌어지는 이야기",
    "director": "봉준호",
    "actors": "송강호, 이선균, 조여정, 최우식, 박소담",
    "release_year": "2019",
}

# 쿼리 생성
queries = query_generator.generate_queries(movie_data)
queries

[{'query': '백수 가족이 부자집에 취업하는 영화', 'query_type': 'plot', 'language': 'ko'},
 {'query': '송강호 이선균 나오는 영화', 'query_type': 'actor', 'language': 'ko'},
 {'query': '봉준호 감독 드라마', 'query_type': 'director', 'language': 'ko'},
 {'query': '드라마 스릴러 코미디', 'query_type': 'genre', 'language': 'ko'},
 {'query': '가족과 계급 다룬 영화', 'query_type': 'mood', 'language': 'ko'},
 {'query': '2019년 봉준호 감독 드라마', 'query_type': 'hybrid', 'language': 'ko'},
 {'query': '전원 백수 취업 이야기', 'query_type': 'plot', 'language': 'ko'}]

In [5]:
from data_scraping.common import load_movie_data

movie_data = load_movie_data()

In [6]:
import pandas as pd

# 모든 열이 보이도록 pandas 옵션 설정
pd.set_option("display.max_columns", None)
movie_data.head(1)

,movie_id,title,genres,imdb_id,tmdb_id,adult,backdrop_path,id,title_tmdb,original_title,overview,poster_path,media_type,original_language,genre_ids,popularity,release_date,video,vote_average,vote_count,genres_tmdb,language,total_title
0,292731,The Monroy Affaire (2022),Drama,tt26812510,1032473.0,False,/meFyMj4e1riRPEQTT2XvKJdYTFh.jpg,1032473,El caso Monroy,El caso Monroy,,/eZzAPq72NhCZIgUzzmu3CWJdKCr.jpg,movie,es,[18],1.316,2022-10-06,False,10.0,1.0,드라마,스페인어,El caso Monroy


In [12]:
# movie_data.iloc[0]에서 데이터 추출하여 QueryGenerator 입력 형식으로 변환
row = movie_data.iloc[6]

# QueryGenerator에 필요한 형식으로 데이터 변환
movie_input = {
    "title": row.get("title", row.get("title_tmdb", row.get("total_title", "Unknown"))),
    "original_title": row.get("original_title", "Unknown"),
    "genres": row.get("genres_tmdb", row.get("genres", "Unknown")),
    "overview": (
        row.get("overview", "No overview available")
        if row.get("overview")
        else "줄거리 정보가 없습니다."
    ),
    "director": row.get("director", "Unknown"),
    "actors": row.get("actors", "Unknown"),
    "keywords": row.get("keywords", "None"),
    "release_year": (
        str(row.get("release_date", "Unknown"))[:4]
        if row.get("release_date")
        else "Unknown"
    ),
}

print("📝 변환된 영화 데이터:")
print("=" * 60)
for key, value in movie_input.items():
    print(f"{key}: {value}")
print("=" * 60)

📝 변환된 영화 데이터:
title: Son of Sinbad (1955)
original_title: Son of Sinbad
genres: 모험 판타지
overview: 줄거리 정보가 없습니다.
director: Unknown
actors: Unknown
keywords: None
release_year: 1955


In [13]:
# QueryGenerator로 쿼리 생성
queries = query_generator.generate_queries(movie_input)
queries

[{'query': '1955년 모험 판타지 영화', 'query_type': 'hybrid', 'language': 'ko'},
 {'query': '모험 판타지 영화', 'query_type': 'genre', 'language': 'ko'},
 {'query': '1955년 옛날 이야기 영화', 'query_type': 'mood', 'language': 'ko'},
 {'query': '모험 판타지 옛날 이야기 영화', 'query_type': 'hybrid', 'language': 'ko'},
 {'query': '1955년 모험 판타지 옛날 이야기', 'query_type': 'hybrid', 'language': 'ko'},
 {'query': '1955년 옛날 모험 영화', 'query_type': 'mood', 'language': 'ko'},
 {'query': '모험 판타지 옛날 이야기 영화 1955', 'query_type': 'hybrid', 'language': 'ko'}]